# SDC Manifest Builder

Reads **only the first and last records** of each SDC file to extract `t_min` / `t_max`  
without loading the full point cloud into RAM.

SDC timestamps are in **SOW (Seconds of Week)**. This notebook converts them to  
**GPS seconds** using: `t_gps = t_sow + day_shift * 86400 + leapsec`  
so that manifest values are directly comparable to `georef_time_window` bounds.

Outputs one CSV per scanner:
- `manifest_HA.csv`
- `manifest_LR.csv`
- `manifest_PUCK.csv`

These manifests are then referenced in the scanner YAML configs so that  
`pointCloudGeoref.py` only processes SDC files that overlap the time window of interest.

In [1]:
import struct
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

## Configuration

In [2]:
# --- Scanner SDC directories ---
SCANNER_DIRS = {
    "HA":   Path("/media/b085164/Elements/CALIB_26_02_25/VUX_SDC/HA/"),
    "LR":   Path("/media/b085164/Elements/CALIB_26_02_25/VUX_SDC/LR/"),
    "PUCK": Path("/media/b085164/Elements/CALIB_26_02_25/PUCK_SDC/"),
}

# --- Time corrections (SOW -> GPS seconds) ---
# SDC timestamps are Seconds of Week (SOW).
# GPS seconds = SOW  +  day_shift * 86400  +  leapsec
#
# day_shift : number of whole days elapsed since Sunday of the GPS week.
#             Monday=1, Tuesday=2, ... Sunday=0
#             Check your acquisition date and set accordingly.
# leapsec   : GPS-UTC leap seconds (18 since 2017-01-01, as in scanner YAMLs)
#
# HA and LR are synchronised so they share the same correction.
TIME_CORRECTIONS = {
    #          day_shift  leapsec
    "HA":   {"day_shift": 3, "leapsec": 18},
    "LR":   {"day_shift": 3, "leapsec": 18},
    "PUCK": {"day_shift": 0, "leapsec": 18},
}

# --- Where to write the manifests ---
MANIFEST_DIR = Path("/media/b085164/Elements/CALIB_26_02_25/manifests")
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# --- How many records to read at each end to estimate t_min / t_max ---
# 500 records = ~20 KB per file, negligible RAM, robust against edge artifacts
N_EDGE = 500

## SDC record layout

In [3]:
RECORD_DTYPE = np.dtype([
    ("t",    "<f8"),  # field 0 — GPS time
    ("f1",   "<f4"),  # field 1
    ("f2",   "<f4"),  # field 2
    ("x",    "<f4"),  # field 3
    ("y",    "<f4"),  # field 4
    ("z",    "<f4"),  # field 5
    ("u6",   "<u2"),
    ("u7",   "<u2"),
    ("u8",   "u1"),
    ("u9",   "u1"),
    ("u10",  "u1"),
    ("u11",  "<u2"),
    ("u12",  "u1"),
    ("u13",  "u1"),
    ("f14",  "<f4"),
    ("u15",  "<u2"),
])
RECORD_SIZE = RECORD_DTYPE.itemsize
print(f"Record size: {RECORD_SIZE} bytes")

Record size: 45 bytes


## Core function — read only edges of one SDC

In [4]:
def sow_to_gps(t_sow: np.ndarray, day_shift: int, leapsec: float) -> np.ndarray:
    """
    Convert SDC Seconds-of-Week timestamps to GPS seconds.
        t_gps = t_sow + day_shift * 86400 + leapsec
    """
    return t_sow + day_shift * 86400.0 + leapsec


def sdc_tmin_tmax(
    sdc_path: Path,
    day_shift: int = 0,
    leapsec: float = 0.0,
    n_edge: int = N_EDGE,
) -> dict:
    """
    Extract t_min and t_max from an SDC file by reading only
    the first and last `n_edge` records.

    Timestamps are converted from SOW to GPS seconds:
        t_gps = t_sow + day_shift * 86400 + leapsec

    Returns a dict with keys:
        filename    : str
        t_min_sow   : float  (raw SOW)
        t_max_sow   : float  (raw SOW)
        t_min       : float  (GPS seconds, after correction)
        t_max       : float  (GPS seconds, after correction)
        n_records   : int    (total point count)
        error       : str or None
    """
    result = {
        "filename":  sdc_path.name,
        "t_min_sow": np.nan,
        "t_max_sow": np.nan,
        "t_min":     np.nan,
        "t_max":     np.nan,
        "n_records": 0,
        "error":     None,
    }

    try:
        file_size = sdc_path.stat().st_size

        # --- read header size (first 4 bytes) ---
        with open(sdc_path, "rb") as f:
            header_bytes = f.read(4)
        size_of_header = struct.unpack("<I", header_bytes)[0]

        payload_size = file_size - size_of_header
        if payload_size <= 0 or payload_size % RECORD_SIZE != 0:
            result["error"] = (
                f"Invalid payload: {payload_size} bytes "
                f"(header={size_of_header}, record={RECORD_SIZE})"
            )
            return result

        n_records = payload_size // RECORD_SIZE
        result["n_records"] = int(n_records)

        n_read = min(n_edge, n_records)

        with open(sdc_path, "rb") as f:
            # --- first n_read records ---
            f.seek(size_of_header)
            head = np.frombuffer(f.read(n_read * RECORD_SIZE), dtype=RECORD_DTYPE)
            t_head = head["t"]

            # --- last n_read records (only if file is large enough) ---
            if n_records > n_read:
                tail_offset = size_of_header + (n_records - n_read) * RECORD_SIZE
                f.seek(tail_offset)
                tail = np.frombuffer(f.read(n_read * RECORD_SIZE), dtype=RECORD_DTYPE)
                t_tail = tail["t"]
            else:
                t_tail = t_head  # small file, head covers everything

        # filter out garbage values before taking min/max
        t_all = np.concatenate([t_head, t_tail])
        mask = np.isfinite(t_all) & (t_all > 0) & (t_all < 1e8)
        if not np.any(mask):
            result["error"] = "No finite timestamps found in edge records"
            return result

        t_valid = t_all[mask]
        t_min_sow = float(np.min(t_valid))
        t_max_sow = float(np.max(t_valid))

        result["t_min_sow"] = t_min_sow
        result["t_max_sow"] = t_max_sow
        result["t_min"]     = float(sow_to_gps(np.array([t_min_sow]), day_shift, leapsec)[0])
        result["t_max"]     = float(sow_to_gps(np.array([t_max_sow]), day_shift, leapsec)[0])

    except Exception as e:
        result["error"] = str(e)

    return result


# Quick sanity check on one file (update path to an actual SDC on your machine)
# test = sdc_tmin_tmax(
#     Path("/media/b085164/Elements/CALIB_26_02_25/VUX_SDC/HA/260225_124306_VUX-HA1.sdc"),
#     day_shift=TIME_CORRECTIONS["HA"]["day_shift"],
#     leapsec=TIME_CORRECTIONS["HA"]["leapsec"],
# )
# print(test)

## Build manifests for all scanners

In [5]:
manifest_paths = {}

for scanner_name, sdc_dir in SCANNER_DIRS.items():
    print(f"\n{'='*60}")
    print(f"Scanner: {scanner_name}  —  {sdc_dir}")
    print(f"{'='*60}")

    if not sdc_dir.exists():
        print(f"  [SKIP] Directory not found: {sdc_dir}")
        continue

    sdc_files = sorted(sdc_dir.glob("*.sdc"))
    if not sdc_files:
        print(f"  [SKIP] No .sdc files found in {sdc_dir}")
        continue

    tc = TIME_CORRECTIONS[scanner_name]
    day_shift = tc["day_shift"]
    leapsec   = tc["leapsec"]
    print(f"  Found {len(sdc_files)} SDC files")
    print(f"  Time correction: day_shift={day_shift}, leapsec={leapsec}  "
          f"=> offset = {day_shift * 86400 + leapsec} s")

    rows = []
    errors = []

    for sdc_path in tqdm(sdc_files, desc=scanner_name, unit="file"):
        info = sdc_tmin_tmax(sdc_path, day_shift=day_shift, leapsec=leapsec)
        rows.append(info)
        if info["error"]:
            errors.append((sdc_path.name, info["error"]))

    df = pd.DataFrame(rows, columns=["filename", "t_min_sow", "t_max_sow", "t_min", "t_max", "n_records", "error"])
    df = df.sort_values("t_min").reset_index(drop=True)

    out_path = MANIFEST_DIR / f"manifest_{scanner_name}.csv"
    df_out = df[["filename", "t_min_sow", "t_max_sow", "t_min", "t_max", "n_records"]].copy()
    df_out.to_csv(out_path, index=False, float_format="%.6f")
    manifest_paths[scanner_name] = out_path

    print(f"  Saved: {out_path}")
    print(f"  SOW range : [{df['t_min_sow'].min():.3f}, {df['t_max_sow'].max():.3f}]")
    print(f"  GPS range : [{df['t_min'].min():.3f}, {df['t_max'].max():.3f}]")

    if errors:
        print(f"  ERRORS ({len(errors)}):")
        for fname, err in errors:
            print(f"    {fname}: {err}")

print("\nDone.")


Scanner: HA  —  /media/b085164/Elements/CALIB_26_02_25/VUX_SDC/HA
  Found 28 SDC files
  Time correction: day_shift=3, leapsec=18  => offset = 259218 s


HA:   0%|          | 0/28 [00:00<?, ?file/s]

  Saved: /media/b085164/Elements/CALIB_26_02_25/manifests/manifest_HA.csv
  SOW range : [45787.361, 47432.773]
  GPS range : [305005.361, 306650.773]

Scanner: LR  —  /media/b085164/Elements/CALIB_26_02_25/VUX_SDC/LR
  Found 28 SDC files
  Time correction: day_shift=3, leapsec=18  => offset = 259218 s


LR:   0%|          | 0/28 [00:00<?, ?file/s]

  Saved: /media/b085164/Elements/CALIB_26_02_25/manifests/manifest_LR.csv
  SOW range : [45787.129, 47432.774]
  GPS range : [305005.129, 306650.774]

Scanner: PUCK  —  /media/b085164/Elements/CALIB_26_02_25/PUCK_SDC
  Found 2 SDC files
  Time correction: day_shift=0, leapsec=18  => offset = 18 s


PUCK:   0%|          | 0/2 [00:00<?, ?file/s]

  Saved: /media/b085164/Elements/CALIB_26_02_25/manifests/manifest_PUCK.csv
  SOW range : [304977.781, 306637.769]
  GPS range : [304995.781, 306655.769]

Done.


## Preview manifests

In [6]:
for scanner_name, csv_path in manifest_paths.items():
    df = pd.read_csv(csv_path)
    print(f"\n--- {scanner_name} ({len(df)} files) ---")
    display(df.head(10))


--- HA (28 files) ---


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,260225_124306_VUX-HA1.sdc,45787.361117,46095.553780,305005.361117,305313.553780,219446068
1,260225_124815_VUX-HA1.sdc,46095.555896,46157.883835,305313.555896,305375.883835,36385331
2,260225_125015_VUX-HA1.sdc,46215.946256,46487.927875,305433.946256,305705.927875,185558813
3,260225_125500_VUX-HA1.sdc,46500.806100,46518.402443,305718.806100,305736.402443,13087821
4,260225_125518_VUX-HA1.sdc,46519.201321,46523.913825,305737.201321,305741.913825,3150195
5,260225_125525_VUX-HA1.sdc,46525.600801,46542.632882,305743.600801,305760.632882,11644239
6,260225_125543_VUX-HA1.sdc,46544.111174,46549.302802,305762.111174,305767.302802,4060132
7,260225_125550_VUX-HA1.sdc,46550.905950,46555.088245,305768.905950,305773.088245,2835482
8,260225_125555_VUX-HA1.sdc,46556.486158,46567.231700,305774.486158,305785.231700,7240099
9,260225_125607_VUX-HA1.sdc,46568.421082,46575.099513,305786.421082,305793.099513,4801453



--- LR (28 files) ---


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,260225_124306_VUX1-LR.sdc,45787.128625,46095.526568,305005.128625,305313.526568,183841319
1,260225_124815_VUX1-LR.sdc,46095.528638,46157.881009,305313.528638,305375.881009,28778753
2,260225_125015_VUX1-LR.sdc,46216.048931,46487.924359,305434.048931,305705.924359,162908156
3,260225_125500_VUX1-LR.sdc,46501.018082,46518.395526,305719.018082,305736.395526,12173366
4,260225_125518_VUX1-LR.sdc,46519.018681,46523.910708,305737.018681,305741.910708,2975152
5,260225_125525_VUX1-LR.sdc,46525.533696,46542.636597,305743.533696,305760.636597,10045110
6,260225_125543_VUX1-LR.sdc,46544.008198,46549.305014,305762.008198,305767.305014,4120361
7,260225_125550_VUX1-LR.sdc,46550.743619,46555.086725,305768.743619,305773.086725,2614199
8,260225_125555_VUX1-LR.sdc,46556.398724,46567.229555,305774.398724,305785.229555,6833225
9,260225_125607_VUX1-LR.sdc,46568.423707,46575.101038,305786.423707,305793.101038,3887905



--- PUCK (2 files) ---


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,lidar_20260225_124315.sdc,304977.781038,305362.920062,304995.781038,305380.920062,45459189
1,lidar_20260225_125030.sdc,305413.187172,306637.768712,305431.187172,306655.768712,161935392


## (Optional) Filter preview — which files fall in a time window?

Useful to verify before running the full georef pipeline.

In [7]:
# --- Preview: which SDCs overlap the outage window? ---
outage_t_start = 305120.0
outage_duration = 475.0
margin_s = 60.0

t_lo = outage_t_start - margin_s
t_hi = outage_t_start + outage_duration + margin_s

print(f"Window: [{t_lo:.1f}, {t_hi:.1f}]  ({(t_hi - t_lo):.0f} s)")
print()

for scanner_name, csv_path in manifest_paths.items():
    df = pd.read_csv(csv_path)
    # a file overlaps the window if its t_max >= t_lo AND t_min <= t_hi
    mask = (df["t_max"] >= t_lo) & (df["t_min"] <= t_hi)
    print(f"  {scanner_name}: {mask.sum()}/{len(df)} files overlap the window")
    display(df[mask])

Window: [305060.0, 305655.0]  (595 s)

  HA: 3/28 files overlap the window


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,260225_124306_VUX-HA1.sdc,45787.361117,46095.553780,305005.361117,305313.553780,219446068
1,260225_124815_VUX-HA1.sdc,46095.555896,46157.883835,305313.555896,305375.883835,36385331
2,260225_125015_VUX-HA1.sdc,46215.946256,46487.927875,305433.946256,305705.927875,185558813


  LR: 3/28 files overlap the window


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,260225_124306_VUX1-LR.sdc,45787.128625,46095.526568,305005.128625,305313.526568,183841319
1,260225_124815_VUX1-LR.sdc,46095.528638,46157.881009,305313.528638,305375.881009,28778753
2,260225_125015_VUX1-LR.sdc,46216.048931,46487.924359,305434.048931,305705.924359,162908156


  PUCK: 2/2 files overlap the window


,filename,t_min_sow,t_max_sow,t_min,t_max,n_records
0,lidar_20260225_124315.sdc,304977.781038,305362.920062,304995.781038,305380.920062,45459189
1,lidar_20260225_125030.sdc,305413.187172,306637.768712,305431.187172,306655.768712,161935392
